# Results overview

This notebook reads the result files written by the pipeline (`results/metrics`, `results/tables`) and shows the tables behind the manuscript. It needs no microdata. Run the scripts first if `results/` is empty.

In [1]:
from pathlib import Path
import pandas as pd
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
MET, TAB = ROOT / 'results' / 'metrics', ROOT / 'results' / 'tables'

## Sample and prevalence by year

In [2]:
pd.read_csv(TAB / 'T1_sample_by_year.csv').round(4)

,year,n,non_enrolled,prev,lo_boot,hi_boot,incident,persistent,weighted_share_persistent_among_known
0,2017,9892,494,0.0451,0.0396,0.0512,143.0,351.0,0.7074
1,2018,10547,467,0.0389,0.0344,0.0436,146.0,321.0,0.6769
2,2019,9570,421,0.0370,0.0316,0.0422,118.0,303.0,0.6935
3,2020,10022,460,0.0464,0.0394,0.0524,73.0,94.0,0.5046
4,2021,9169,478,0.0494,0.0426,0.0567,144.0,334.0,0.6716
5,2022,9127,379,0.0354,0.0307,0.0411,111.0,268.0,0.7268
6,2023,8997,297,0.0287,0.0244,0.0330,96.0,201.0,0.7060
7,2024,8576,304,0.0312,0.0265,0.0366,99.0,205.0,0.6809
8,2025,8602,307,0.0298,0.0253,0.0343,116.0,191.0,0.6293


## E1-E2: performance of the frozen models by evaluation set (unweighted training)

In [3]:
p = pd.read_csv(MET / 'E1_E2_performance.csv')
p[p.training == 'unweighted'].pivot(index='test_set', columns='model', values=['auc', 'prauc', 'citl', 'slope']).round(3)

auc                prauc                 citl                slope              
model            LR     RF    XGB     LR     RF    XGB     LR     RF    XGB     LR     RF    XGB
test_set                                                                                        
2020          0.821  0.831  0.837  0.396  0.429  0.452  0.366  0.369  0.412  0.850  0.792  0.809
2021          0.887  0.904  0.899  0.567  0.597  0.590  0.382  0.418  0.430  1.136  1.056  0.997
2022          0.924  0.937  0.937  0.545  0.585  0.583 -0.014  0.035  0.030  1.262  1.260  1.142
2023          0.909  0.929  0.928  0.510  0.564  0.563 -0.125 -0.043 -0.055  1.279  1.284  1.174
2024          0.868  0.898  0.883  0.486  0.505  0.520 -0.069  0.000  0.011  1.123  1.094  1.025
2025          0.869  0.883  0.888  0.485  0.524  0.541 -0.092  0.053  0.043  1.141  1.070  1.075
CV 2017-2019  0.918  0.926  0.930  0.476  0.508  0.526 -0.035 -0.030 -0.045  1.026  1.115  1.012

### Frozen model against a model cross-validated within each test year

In [4]:
r = pd.read_csv(MET / 'E2_in_year_reference.csv')
r[r.metric == 'auc'].round(3)

,year,model,metric,in_year_cv,in_year_lo,in_year_hi,frozen,frozen_minus_in_year,lo,hi
0,2020,LR,auc,0.828,0.790,0.863,0.821,-0.007,-0.018,0.003
2,2020,RF,auc,0.831,0.799,0.862,0.831,-0.001,-0.026,0.018
4,2020,XGB,auc,0.844,0.812,0.875,0.837,-0.007,-0.027,0.010
6,2021,LR,auc,0.898,0.868,0.928,0.887,-0.011,-0.022,-0.002
8,2021,RF,auc,0.905,0.875,0.932,0.904,-0.002,-0.015,0.010
10,2021,XGB,auc,0.899,0.865,0.930,0.899,0.001,-0.008,0.009
12,2022,LR,auc,0.931,0.912,0.947,0.924,-0.006,-0.014,0.002
14,2022,RF,auc,0.936,0.921,0.951,0.937,0.001,-0.008,0.010
16,2022,XGB,auc,0.937,0.918,0.954,0.937,0.000,-0.008,0.009
18,2023,LR,auc,0.911,0.881,0.936,0.909,-0.003,-0.009,0.004


## E3: updating strategies

In [5]:
u = pd.read_csv(MET / 'E3_update_2025.csv')
u[['model', 'strategy', 'auc', 'prauc', 'brier', 'citl', 'citl_lo', 'citl_hi', 'slope']].round(3)

,model,strategy,auc,prauc,brier,citl,citl_lo,citl_hi,slope
0,LR,Frozen (2017-2019),0.869,0.485,0.020,-0.092,-0.263,0.063,1.141
1,LR,Intercept recalibration,0.869,0.485,0.021,-0.025,-0.196,0.130,1.141
2,LR,Intercept + slope recalibration,0.869,0.485,0.020,-0.003,-0.197,0.170,0.935
3,LR,Retrained on 2022-2024,0.884,0.507,0.020,0.068,-0.115,0.235,1.096
4,LR,Retrained on 2017-2024,0.878,0.497,0.020,-0.096,-0.272,0.062,1.128
5,RF,Frozen (2017-2019),0.883,0.524,0.020,0.053,-0.115,0.202,1.070
6,RF,Intercept recalibration,0.883,0.524,0.020,0.054,-0.114,0.203,1.070
7,RF,Intercept + slope recalibration,0.883,0.524,0.019,0.085,-0.108,0.255,0.890
8,RF,Retrained on 2022-2024,0.889,0.540,0.019,0.054,-0.124,0.213,1.033
9,RF,Retrained on 2017-2024,0.890,0.539,0.019,-0.016,-0.187,0.137,1.067


In [6]:
ro = pd.read_csv(MET / 'E3_rolling_origin.csv')
ro[ro.model == 'XGB'].pivot(index='strategy', columns='year', values='citl').round(2)

year,2020,2021,2022,2023,2024,2025
strategy,,,,,,
Frozen (2017-2019),0.41,0.43,0.03,-0.06,0.01,0.04
Intercept + slope recalibration on t-1,0.49,0.03,-0.40,-0.07,0.08,0.03
Intercept recalibration on t-1,0.50,0.02,-0.40,-0.09,0.07,0.03
"Retrained, all previous years",NaN,0.32,-0.11,-0.18,-0.06,-0.00
"Retrained, last three years",NaN,0.32,-0.21,-0.31,-0.08,0.11


## E4: subgroups (XGBoost)

In [7]:
s = pd.read_csv(MET / 'E4_subgroups.csv')
s[(s.model == 'XGB') & ~s.group.str.contains(' x ')][['period', 'group', 'events', 'auc', 'citl', 'citl_lo', 'citl_hi', 'recall', 'precision']].round(3)

,period,group,events,auc,citl,citl_lo,citl_hi,recall,precision
2,In-period CV 2017-2019,All,1382,0.930,-0.045,-0.149,0.051,0.744,0.300
5,In-period CV 2017-2019,Urban,544,0.932,-0.056,-0.222,0.092,0.709,0.296
8,In-period CV 2017-2019,Rural,838,0.919,-0.030,-0.165,0.083,0.788,0.305
11,In-period CV 2017-2019,Non-poor,719,0.933,-0.022,-0.155,0.104,0.705,0.296
14,In-period CV 2017-2019,Poor,663,0.908,-0.071,-0.230,0.078,0.785,0.304
17,In-period CV 2017-2019,Spanish,1159,0.931,-0.033,-0.149,0.069,0.747,0.305
20,In-period CV 2017-2019,Indigenous tongue,223,0.919,-0.140,-0.357,0.064,0.711,0.259
23,In-period CV 2017-2019,Male,671,0.927,-0.112,-0.264,0.027,0.747,0.268
26,In-period CV 2017-2019,Female,711,0.932,0.026,-0.116,0.168,0.741,0.337
98,Pandemic 2020-2021,All,938,0.869,0.421,0.288,0.552,0.616,0.346


## E5: attribution by predictor family

In [8]:
pd.read_csv(MET / 'E5_shap_family.csv').round(3)

,family,Training 2017-2019,Pandemic 2020-2021,Post-pandemic 2022-2025,2020,2021,2022,2023,2024,2025
0,Age and schooling trajectory,1.616,1.578,1.474,1.594,1.563,1.509,1.488,1.464,1.438
1,Other individual,0.161,0.151,0.155,0.152,0.151,0.155,0.152,0.157,0.158
2,Household socioeconomic,0.358,0.362,0.361,0.366,0.359,0.357,0.358,0.364,0.365
3,Digital access,0.181,0.185,0.197,0.178,0.192,0.194,0.196,0.199,0.197
4,Area,0.209,0.201,0.201,0.201,0.201,0.201,0.202,0.200,0.202


## E6: ablation and type of outcome

In [9]:
a = pd.read_csv(MET / 'E6_ablation.csv')
a[a.model == 'XGB'].pivot(index='predictor_set', columns='year', values='auc').round(3)

year,2020,2021,2022,2023,2024,2025
predictor_set,,,,,,
Age and schooling trajectory only,0.793,0.871,0.896,0.888,0.873,0.862
Full + fixed internet connection,0.839,0.899,0.938,0.928,0.884,0.888
Full + household internet item (not comparable from 2023),0.839,0.899,0.938,0.927,0.883,0.888
Full model,0.837,0.899,0.937,0.928,0.883,0.888
Household and area only,0.724,0.766,0.797,0.791,0.727,0.778
Without age and schooling trajectory,0.731,0.767,0.810,0.806,0.733,0.780
Without age-grade distortion,0.845,0.897,0.940,0.930,0.884,0.891
Without digital access,0.833,0.898,0.934,0.927,0.884,0.884
Without schooling trajectory,0.771,0.794,0.848,0.849,0.787,0.806


In [10]:
pd.read_csv(TAB / 'T5_outcome_validity.csv').round(3)

,block,outcome,events,prev,auc,auc_lo,auc_hi,prauc,prauc_lo,prauc_hi,source
0,"In-period, 2017--2019",Not enrolled (status),1382,0.040,0.930,0.919,0.940,0.526,0.487,0.564,E1_E2_performance.csv
1,"In-period, 2017--2019",\quad persistent,975,0.028,0.955,0.945,0.964,0.566,0.516,0.610,leakage_outcome_types.csv
2,"In-period, 2017--2019",\quad incident,407,0.013,0.871,0.846,0.894,0.117,0.087,0.161,leakage_outcome_types.csv
3,"Frozen model, 2022--2025",Not enrolled (status),1287,0.031,0.910,0.891,0.926,0.552,0.513,0.592,leakage_outcome_types.csv
4,"Frozen model, 2022--2025",\quad persistent,865,0.022,0.964,0.948,0.975,0.647,0.604,0.693,leakage_outcome_types.csv
5,"Frozen model, 2022--2025",\quad incident,422,0.010,0.790,0.749,0.831,0.072,0.054,0.098,leakage_outcome_types.csv
6,"Frozen model, 2022--2025","\quad incident, own model$^{\mathrm{a}}$",422,0.010,0.787,0.745,0.825,0.090,0.066,0.126,leakage_incident_model.csv
7,"Not enrolled next year (panel, baseline 2020--...",\quad status score,168,0.013,0.815,0.769,0.853,0.064,0.043,0.103,panel_prospective_performance.csv
8,"Not enrolled next year (panel, baseline 2020--...",\quad prospective model$^{\mathrm{b}}$,168,0.013,0.817,0.774,0.858,0.072,0.051,0.103,panel_prospective_performance.csv


## Where every number of the manuscript comes from

In [11]:
reg = pd.read_csv(TAB / 'manuscript_numbers.csv')
print(len(reg), 'numbers registered')
reg.sample(8, random_state=1)

6537 numbers registered


,key,value,used_in_manuscript,source,selector
5287,sub.post.LR.indig.flag_rate.ci,"0.047, 0.071",False,results/metrics/E4_subgroups.csv,period=Post-pandemic 2022-2025; model=LR; grou...
5754,sub.post.XGB.poor.precision,0.350,False,results/metrics/E4_subgroups.csv,period=Post-pandemic 2022-2025; model=XGB; gro...
2377,abl.XGB.noover.dauc.nsig,1,False,results/metrics/E6_ablation_paired.csv,model=XGB; predictor_set=Without age-grade dis...
5144,sub.post.LR.urban.citl,$-$0.257,False,results/metrics/E4_subgroups.csv,period=Post-pandemic 2022-2025; model=LR; grou...
1637,upd.RF.intslope.citl.2,0.08,False,results/metrics/E3_update_2025.csv,model=RF; strategy=Intercept + slope recalibra...
354,perfw.LR.2022.citl,0.030,False,results/metrics/E1_E2_performance.csv,training=weighted; model=LR; test_set=2022; co...
6363,grec.XGB.frozen.spanish.recall,0.678,False,results/metrics/E4_group_recalibration.csv,model=XGB; variant=Frozen; group=Spanish; colu...
712,perf.RF.2024.prauc,0.505,False,results/metrics/E1_E2_performance.csv,training=unweighted; model=RF; test_set=2024; ...
